# YOLOv12 — NXP Cup 2026 (GPU training on Colab)

Trains the tiny-object sign detector on a **free Colab GPU** and lets you
download the weights (`best.pt` + `best.onnx`).

**First: turn on the GPU** → *Runtime → Change runtime type → Hardware
accelerator: T4 GPU → Save*. Then *Runtime → Run all*.

Key facts baked into the code: images are **512×512**, objects are **tiny**
(median ~15 px) so we train at **512** and never downscale, and the output
head has **9 classes** (`A B C Left Right Straight X Y Z`).

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'No GPU — enable it via Runtime > Change runtime type > T4 GPU'

## 2. Get the code + dataset

Clones this branch. The dataset ships inside the repo as `nxpcup_dataset.zip`,
so nothing else to upload.

> **Private repo?** Put a GitHub token in `TOKEN` below (a fine-grained PAT
> with read access). Leave it blank if the repo is public.

In [ ]:
REPO   = 'github.com/saquibjawedbit/grabber.git'
BRANCH = 'claude/yolov12-edge-device-model-ctamub'
TOKEN  = ''  # optional: paste a GitHub PAT here if the repo is private

url = f'https://{TOKEN + "@" if TOKEN else ""}{REPO}'
!git clone --branch $BRANCH --single-branch $url 2>&1 | tail -3
%cd grabber/yolov12-training
!unzip -q -o nxpcup_dataset.zip -d dataset
import os
for s in ['train','valid','test']:
    n = len(os.listdir(f'dataset/NXPCUP_2026.v2-v1_a.yolov12/{s}/images'))
    print(f'{s}: {n} images')

## 3. Install Ultralytics

Colab already has a CUDA build of PyTorch, so we only add Ultralytics + the
ONNX export tools.

In [ ]:
!pip install -q ultralytics onnx onnxruntime-gpu onnxslim
import torch, ultralytics
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('ultralytics', ultralytics.__version__)

## 4. Train (full run on GPU)

Uses the tuned config (`configs/train_nano.yaml`): YOLO12n @ 512, tiny-object-safe
augmentation, `fliplr` OFF (Left/Right are directional), cosine LR, mosaic closed
for the last epochs. Tweak the three knobs below if you want.

- `MODEL`  — `yolo12n.pt` (realtime edge default) or `yolo12s.pt` (more accurate,
  helps the smallest signs, ~3× compute).
- `EPOCHS` — 200 is plenty here; early-stopping (patience 40) usually ends sooner.
- `IMGSZ`  — keep **512**. Lower erases the ~15 px signs.

In [ ]:
MODEL  = 'yolo12n.pt'
EPOCHS = 200
IMGSZ  = 512

!python scripts/train.py --model {MODEL} --epochs {EPOCHS} --imgsz {IMGSZ} --device 0 --cache ram --name nxpcup_gpu

## 5. Validate on the held-out test split

In [ ]:
!python scripts/validate.py --weights runs/detect/nxpcup_gpu/weights/best.pt --imgsz {IMGSZ} --split test

## 6. Look at the results

In [ ]:
from IPython.display import Image, display
run = 'runs/detect/nxpcup_gpu'
for p in ['results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png']:
    import os
    fp = os.path.join(run, p)
    if os.path.exists(fp):
        print(p); display(Image(filename=fp))

## 7. Export for the buggy

ONNX runs everywhere. On a Jetson, also export a TensorRT engine **on the
Jetson itself** (engines aren't portable across devices) with:
`python scripts/export.py --weights best.pt --format engine --half --imgsz 512`.

In [ ]:
!python scripts/export.py --weights runs/detect/nxpcup_gpu/weights/best.pt --format onnx --imgsz {IMGSZ} --simplify

## 8. Download the weights

Bundles `best.pt`, `best.onnx`, and the plots into one zip and downloads it.

In [ ]:
import shutil, os
os.makedirs('download', exist_ok=True)
run = 'runs/detect/nxpcup_gpu'
for f in ['weights/best.pt', 'weights/best.onnx', 'results.csv',
          'results.png', 'confusion_matrix.png', 'args.yaml']:
    src = os.path.join(run, f)
    if os.path.exists(src):
        shutil.copy(src, 'download/')
shutil.make_archive('nxpcup_yolov12_weights', 'zip', 'download')
print('bundled:', os.listdir('download'))
try:
    from google.colab import files
    files.download('nxpcup_yolov12_weights.zip')
except Exception as e:
    print('Not on Colab or download blocked — grab nxpcup_yolov12_weights.zip from the file browser.', e)